# 🔬 PHÒNG THÍ NGHIỆM AI ANOMALY DETECTION (PATCHCORE & MEMORY BANK)
> **Mục tiêu:** Từng bước làm chủ thuật toán AI bắt khuyết tật bo mạch thực tế (DeepPCB) không cần dữ liệu lỗi.
> **Quy trình chuẩn:** Thử nghiệm từng Cell trên Notebook (.ipynb) ➔ Đóng gói thành mã nguồn sản xuất (.py).


## 📦 BƯỚC 1: IMPORT THƯ VIỆN & THIẾT LẬP ĐƯỜNG DẪN DỮ LIỆU
- `cv2`: Đọc ảnh, cắt patch, vẽ bounding box khoanh vùng lỗi.
- `numpy`: Xử lý mảng ma trận vector đặc trưng và khoảng cách Euclidean.
- `matplotlib.pyplot`: Hiển thị ảnh và bản đồ nhiệt (Heatmap) trực quan ngay dưới từng cell.

In [ ]:
# 1. Khai báo các thư viện cần thiết
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 2. Cấu hình đường dẫn tới bộ dữ liệu bo mạch DeepPCB mà ta vừa tải về
DATASET_DIR = os.path.join("..", "datasets", "pcb_defects")
TRAIN_DIR = os.path.join(DATASET_DIR, "train_good")       # 50 ảnh bo mạch chuẩn 100% OK
TEST_DIR = os.path.join(DATASET_DIR, "test_defects")     # 50 ảnh bo mạch bị lỗi thật
ANN_DIR = os.path.join(DATASET_DIR, "annotations")      # Tọa độ lỗi do kỹ sư đánh dấu

print("[*] Thư mục dữ liệu chuẩn OK:", os.path.abspath(TRAIN_DIR))
print("[*] Thư mục dữ liệu lỗi thật:", os.path.abspath(TEST_DIR))


## 🖼️ BƯỚC 2: ĐỌC VÀ HIỂN THỊ ĐỐI CHIẾU (ẢNH CHUẨN vs ẢNH LỖI THẬT)
- Ta lấy cặp ảnh mẫu đầu tiên: `00041000_temp.jpg` (Chuẩn) và `00041000_test.jpg` (Lỗi).
- Trên ảnh lỗi, ta dùng đúng tọa độ vừa học: `(539, 259) -> (592, 316)` để vẽ một khung đỏ khoanh vết đứt mạch.

In [ ]:
# 1. Đọc ảnh mẫu bo mạch chuẩn (Template OK)
img_ok = cv2.imread(os.path.join(TRAIN_DIR, "00041000_temp.jpg"))
img_ok_rgb = cv2.cvtColor(img_ok, cv2.COLOR_BGR2RGB)

# 2. Đọc ảnh mẫu bo mạch bị lỗi thật (Defective Test)
img_ng = cv2.imread(os.path.join(TEST_DIR, "00041000_test.jpg"))
img_ng_rgb = cv2.cvtColor(img_ng, cv2.COLOR_BGR2RGB)

# 3. Vẽ khung đỏ bao quanh vết đứt mạch theo tọa độ ta vừa học:
#    X1 = 539, Y1 = 259 (Góc Trên - Trái)
#    X2 = 592, Y2 = 316 (Góc Dưới - Phải)
#    Chiều Rộng = 592 - 539 = 53px, Chiều Cao = 316 - 259 = 57px
cv2.rectangle(img_ng_rgb, (539, 259), (592, 316), (255, 0, 0), 3)

# 4. Vẽ 2 bức ảnh cạnh nhau để so sánh trực quan bằng Matplotlib
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("BO MACH CHUAN 100% OK (TEMPLATE)", color="green", fontsize=12, fontweight="bold")
plt.imshow(img_ok_rgb)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("BO MACH BI DUT MACH (GROUND TRUTH)", color="red", fontsize=12, fontweight="bold")
plt.imshow(img_ng_rgb)
plt.axis("off")

plt.tight_layout()
plt.show()


: 

## 🧩 BƯỚC 3: CƠ CHẾ PATCHCORE - CẮT ẢNH THÀNH 400 MẢNH GHÉP (PATCHES)
- Kích thước ảnh: 640 x 640 pixel.
- Kích thước mỗi mảnh Patch: 32 x 32 pixel.
- Tổng số mảnh: (640 / 32) x (640 / 32) = 400 mảnh Patch!
- Ta cắt 1 mảnh bình thường và 1 mảnh bị đứt mạch để quan sát dưới góc nhìn của kính lúp AI.

In [ ]:
# 1. Định nghĩa kích thước mảnh ghép
PATCH_SIZE = 32  # 32 x 32 pixel

# 2. Cắt 1 mảnh bình thường trên bo mạch chuẩn (X=128, Y=128)
patch_normal = img_ok_rgb[128:128+PATCH_SIZE, 128:128+PATCH_SIZE]

# 3. Cắt 1 mảnh bị đứt mạch trên bo mạch lỗi (X=544, Y=288 - nằm đúng trong hộp đỏ!)
patch_defect = img_ng_rgb[288:288+PATCH_SIZE, 544:544+PATCH_SIZE]

# 4. Hiển thị cận cảnh 2 mảnh ghép để mắt và AI cùng soi
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.title('MANH CHUAN (32x32px) - Duong dong lien mach', color='green', fontweight='bold')
plt.imshow(patch_normal)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('MANH BI LOI (32x32px) - Bi dut toac o giua', color='red', fontweight='bold')
plt.imshow(patch_defect)
plt.axis('off')

plt.tight_layout()
plt.show()

print('[*] Buc anh 640x640 duoc chia thanh: (640/32) x (640/32) = 400 manh Patch!')


## BUOC 4: XAY DUNG MEMORY BANK (KHO KY UC CHUAN)
- Doc toan bo 50 anh chuan OK trong `train_good/`.
- Moi anh duoc cat thanh 400 manh Patch 32x32 pixel.
- Moi manh Patch duoc "ep det" thanh vector 3072 chieu (32 x 32 x 3 = 3072).
- Toan bo 50 x 400 = 20.000 vector duoc gom lai thanh bang Memory Bank.
- Memory Bank giong nhu cuon so ghi chep 20.000 dong, moi dong = dac trung 1 manh bo mach binh thuong.

In [ ]:
import time

PATCH_SIZE = 32
NUM_GRIDS  = 640 // PATCH_SIZE  # 640 / 32 = 20 o moi chieu

memory_bank = []

ok_files = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith(".jpg")])
print("[*] Bat dau nap", len(ok_files), "anh chuan OK vao Memory Bank...")

t_start = time.time()

for fname in ok_files:
    img = cv2.imread(os.path.join(TRAIN_DIR, fname))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for row in range(NUM_GRIDS):
        for col in range(NUM_GRIDS):
            y = row * PATCH_SIZE
            x = col * PATCH_SIZE
            patch = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
            # Ep det 32x32x3 = 3072 con so, chuan hoa ve [0, 1]
            vector = patch.flatten().astype(np.float32) / 255.0
            memory_bank.append(vector)

# Chuyen tu list sang ma tran NumPy: shape (20000, 3072)
memory_bank = np.array(memory_bank)

t_end = time.time()

print("[+] MEMORY BANK DA XAY DUNG XONG!")
print("    - So anh chuan OK:", len(ok_files))
print("    - Tong so vector:", memory_bank.shape[0])
print("    - Kich thuoc moi vector:", memory_bank.shape[1], "chieu (32x32x3=3072)")
print("    - Thoi gian xay dung: %.2f giay" % (t_end - t_start))
print("    - Dung luong RAM: %.1f MB" % (memory_bank.nbytes / (1024 * 1024)))

## BUOC 5: SO KHOANG CACH, VE HEATMAP, PHAN QUYET OK/NG
- Quy trinh cho tung manh Patch cua anh loi:
  1. Lay vector dac trung cua Patch do (3072 con so).
  2. Tinh khoang cach Euclidean toi TAT CA 20.000 vector trong Memory Bank.
  3. Lay khoang cach NHO NHAT (Nearest Neighbor).
  4. Khoang cach nho -> Patch binh thuong -> To XANH.
  5. Khoang cach lon -> Patch bat thuong  -> To DO.
- Ghep 400 o lai = ANOMALY HEATMAP -> Phan quyet OK/NG!

In [ ]:
# 1. Doc anh loi that can kiem tra
img_test = cv2.imread(os.path.join(TEST_DIR, "00041000_test.jpg"))
img_test_rgb = cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB)

# 2. VECTORIZATION: Tinh toan song song toan bo 400 Patch cung luc
#    Cong thuc hang dang thuc: (A - B)^2 = A^2 - 2AB + B^2
print("[*] AI dang so sanh 400 Patch voi 20.000 vector bang VECTORIZATION...")
t_start = time.time()

# Cat toan bo 400 Patch thanh ma tran T: Shape (400, 3072)
patches = [img_test_rgb[r * PATCH_SIZE:(r + 1) * PATCH_SIZE, c * PATCH_SIZE:(c + 1) * PATCH_SIZE].flatten().astype(np.float32) / 255.0
           for r in range(NUM_GRIDS) for c in range(NUM_GRIDS)]
T = np.array(patches)  # (400, 3072)
M = memory_bank        # (20000, 3072)

# Khai trien hang dang thuc ma tran:
T_sq = np.sum(T ** 2, axis=1, keepdims=True)            # (400, 1)
M_sq = np.sum(M ** 2, axis=1, keepdims=True).T          # (1, 20000)
dot = np.dot(T, M.T)                                    # Nhan ma tran (400, 20000)

# Ma tran khoang cach binh phuong
dist_sq = np.maximum(T_sq - 2 * dot + M_sq, 0.0)

# Lay khoang cach nho nhat (Nearest Neighbor) cho tung Patch
min_dist = np.sqrt(np.min(dist_sq, axis=1))
anomaly_map = min_dist.reshape(NUM_GRIDS, NUM_GRIDS)

t_end = time.time()
elapsed = t_end - t_start
print(f"[+] VECTORIZATION HOAN TAT!")
print(f"    - Thoi gian so sanh: {elapsed:.4f} giay ({elapsed * 1000:.1f} ms) ⚡⚡⚡")
print(f"    - Nhanh gap ~{54.18 / elapsed:.0f} LAN so voi vong lap for cu!")

# 3. Phong to Heatmap tu 20x20 len 640x640
heatmap = cv2.resize(anomaly_map, (640, 640), interpolation=cv2.INTER_LINEAR)
heatmap_norm = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

# 4. Ghep Heatmap de len anh goc
overlay = cv2.addWeighted(img_test_rgb, 0.5, heatmap_rgb, 0.5, 0)

# 5. Ve ket qua 3 anh canh nhau
plt.figure(figsize=(15, 5))
plt.subplot(1,3,1)
plt.title("ANH LOI (THUC TE)", color="gray", fontweight="bold")
plt.imshow(img_test_rgb); plt.axis("off")
plt.subplot(1,3,2)
plt.title("ANOMALY HEATMAP (Do=Loi, Xanh=OK)", color="red", fontweight="bold")
plt.imshow(heatmap_rgb); plt.axis("off")
plt.subplot(1,3,3)
plt.title("OVERLAY: LOI TO MAU TREN ANH", color="darkorange", fontweight="bold")
plt.imshow(overlay); plt.axis("off")
plt.tight_layout(); plt.show()

# 6. Phan quyet OK / NG
max_score = np.max(anomaly_map)
THRESHOLD = np.percentile(anomaly_map, 90)
print("=" * 50)
print("Diem bat thuong cao nhat:", round(max_score, 4))
print("Nguong phan quyet (top 10%):", round(THRESHOLD, 4))
if max_score > THRESHOLD:
    print("KET QUA: NG - HANG LOI - Phat hien di thuong!")
else:
    print("KET QUA: OK - HANG CHUAN")
print("=" * 50)